# Model 2: Skin Tone Analysis
**Dataset:** Fitzpatrick17k (Kaggle/GitHub)  
**Architecture:** CNN Classifier (MobileNetV3) + K-Means dominant color  
**Output:** `skin_tone_classifier.pth`, `skin_kmeans.pkl`  
**Target:** Accuracy > 85% trên Fitzpatrick scale (6 mức)

---
**Fitzpatrick Scale:**
- Type I: Very fair (always burns)
- Type II: Fair (usually burns)
- Type III: Medium (sometimes burns)
- Type IV: Olive (rarely burns)
- Type V: Brown (very rarely burns)
- Type VI: Dark brown/Black (never burns)

**Dataset download:**
```bash
kaggle datasets download -d google-brain-team/fitzpatrick17k
# hoặc: https://github.com/mattgroh/fitzpatrick17k
```

In [ ]:
# ============================================================
# CELL 1: Install & import
# ============================================================
!pip install torch torchvision scikit-learn pandas pillow matplotlib tqdm requests --quiet

import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: Download Fitzpatrick17k dataset
# ============================================================
import requests

DATA_DIR = "./data/fitzpatrick17k"
os.makedirs(DATA_DIR, exist_ok=True)

CSV_URL = "https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"
CSV_PATH = f"{DATA_DIR}/fitzpatrick17k.csv"

if not os.path.exists(CSV_PATH):
    print("Downloading CSV metadata...")
    r = requests.get(CSV_URL)
    with open(CSV_PATH, 'wb') as f:
        f.write(r.content)
    print("Done.")

df = pd.read_csv(CSV_PATH)
print(f"Total entries: {len(df)}")
print(df.head())
print("\nColumns:", df.columns.tolist())
print("\nFitzpatrick distribution:")
print(df['fitzpatrick_scale'].value_counts().sort_index())

In [ ]:
# ============================================================
# CELL 3: Download images (750 ảnh/class, retry logic)
# ============================================================
import time

IMG_DIR = f"{DATA_DIR}/images"
os.makedirs(IMG_DIR, exist_ok=True)

SAMPLES_PER_CLASS = 750

df_valid = df[df['fitzpatrick_scale'].isin([1, 2, 3, 4, 5, 6])].copy()

df_sampled = df_valid.groupby('fitzpatrick_scale').apply(
    lambda x: x.sample(min(len(x), SAMPLES_PER_CLASS), random_state=42)
).reset_index(drop=True)

print(f"Sampled: {len(df_sampled)} images")
print(df_sampled['fitzpatrick_scale'].value_counts().sort_index())

def download_image(url, save_path, timeout=10, max_retries=2):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200 and 'image' in r.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    f.write(r.content)
                return True
        except:
            if attempt < max_retries - 1:
                time.sleep(0.5)
    return False

downloaded = []
for _, row in tqdm(df_sampled.iterrows(), total=len(df_sampled), desc="Downloading"):
    img_name = f"{row.name}.jpg"
    save_path = f"{IMG_DIR}/{img_name}"
    if os.path.exists(save_path):
        downloaded.append({'path': save_path, 'fitzpatrick': row['fitzpatrick_scale']})
        continue
    if 'url' in row and pd.notna(row['url']):
        success = download_image(row['url'], save_path)
        if success:
            downloaded.append({'path': save_path, 'fitzpatrick': row['fitzpatrick_scale']})

df_local = pd.DataFrame(downloaded)
print(f"\nSuccessfully downloaded: {len(df_local)} images")
print("Per class:")
print(df_local['fitzpatrick'].value_counts().sort_index())

In [ ]:
# ============================================================
# CELL 4: Dataset class (3 class: Light / Medium / Dark)
# ============================================================
# Gộp 6 Fitzpatrick type → 3 class
TONE_MAP = {1: 0, 2: 0, 3: 1, 4: 1, 5: 2, 6: 2}
TONE_LABELS = ['Light', 'Medium', 'Dark']
NUM_CLASSES = 3

df_local['tone'] = df_local['fitzpatrick'].map(TONE_MAP)

class SkinToneDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = int(row['tone'])  # 0=Light, 1=Medium, 2=Dark
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_df, val_df = train_test_split(df_local, test_size=0.2,
                                     stratify=df_local['tone'], random_state=42)

train_ds = SkinToneDataset(train_df, train_transform)
val_ds   = SkinToneDataset(val_df,   val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print("Class distribution (train):", train_df['tone'].value_counts().sort_index().to_dict())
print("Labels:", {i: l for i, l in enumerate(TONE_LABELS)})

In [ ]:
# ============================================================
# CELL 5: Model - EfficientNet-B0 (3 classes)
# ============================================================
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn.functional as F

model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(256, NUM_CLASSES)  # 3 classes
)
model = model.to(DEVICE)

# Class weights để handle imbalanced data
class_counts = train_df['tone'].value_counts().sort_index()
total = class_counts.sum()
weights = torch.tensor([total / (NUM_CLASSES * c) for c in class_counts.values],
                       dtype=torch.float32).to(DEVICE)
print("Class weights:", weights.cpu().numpy().round(3))

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
EPOCHS = 30
warmup  = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine  = CosineAnnealingLR(optimizer, T_max=EPOCHS - 3, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[3])

print(f"EfficientNet-B0 ready — {NUM_CLASSES} classes: {TONE_LABELS}")

In [ ]:
# ============================================================
# CELL 6: Training với early stopping
# ============================================================
best_val_acc = 0.0
patience, patience_counter = 7, 0
history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch:02d} Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        t_loss    += loss.item() * imgs.size(0)
        t_correct += out.argmax(1).eq(labels).sum().item()
        t_total   += imgs.size(0)

    # ---- Validate ----
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += out.argmax(1).eq(labels).sum().item()
            v_total   += imgs.size(0)

    scheduler.step()

    train_acc = t_correct / t_total
    val_acc   = v_correct / v_total
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(v_loss / v_total)

    print(f"Epoch [{epoch:02d}/{EPOCHS}]  "
          f"Train: {train_acc:.4f} ({t_loss/t_total:.4f})  "
          f"Val: {val_acc:.4f} ({v_loss/v_total:.4f})  "
          f"LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'tone_labels': TONE_LABELS,
            'tone_map': TONE_MAP,
            'num_classes': NUM_CLASSES,
            'val_acc': val_acc,
            'epoch': epoch
        }, 'skin_tone_classifier.pth')
        print(f"  --> Saved best (val_acc={val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nBest Val Accuracy: {best_val_acc*100:.1f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_acc'], label='Train'); ax1.plot(history['val_acc'], label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.plot(history['train_loss'], label='Train'); ax2.plot(history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig('training_history.png', dpi=100); plt.show()

In [ ]:
# ============================================================
# CELL 7: K-Means dominant color extractor (3 class)
# ============================================================
SEASON_MAP = {
    'Light':  'Summer',   # Type I + II
    'Medium': 'Autumn',   # Type III + IV
    'Dark':   'Winter'    # Type V + VI
}

COLOR_SUGGESTIONS = {
    'Summer': ['pastel pink', 'lavender', 'soft blue', 'mint', 'rose'],
    'Autumn': ['burnt orange', 'olive green', 'mustard', 'terracotta', 'rust'],
    'Winter': ['black', 'white', 'navy', 'royal blue', 'bright red', 'emerald']
}

def extract_dominant_color(image_path, n_colors=3):
    img = Image.open(image_path).convert('RGB').resize((100, 100))
    pixels = np.array(img).reshape(-1, 3).astype(float)
    kmeans = KMeans(n_clusters=n_colors, n_init=10, random_state=42)
    kmeans.fit(pixels)
    return kmeans.cluster_centers_.astype(int)

# Train KMeans trên 3 cluster (Light/Medium/Dark)
skin_colors = []
skin_labels = []

for _, row in df_local.sample(min(300, len(df_local)), random_state=42).iterrows():
    try:
        img = Image.open(row['path']).convert('RGB').resize((60, 60))
        pixels = np.array(img).reshape(-1, 3).mean(axis=0)
        skin_colors.append(pixels)
        skin_labels.append(int(row['tone']))
    except:
        pass

skin_colors = np.array(skin_colors)
skin_labels = np.array(skin_labels)

kmeans_skin = KMeans(n_clusters=3, n_init=20, random_state=42)
kmeans_skin.fit(skin_colors)

with open('skin_kmeans.pkl', 'wb') as f:
    pickle.dump({
        'kmeans': kmeans_skin,
        'tone_labels': TONE_LABELS,
        'season_map': SEASON_MAP,
        'color_suggestions': COLOR_SUGGESTIONS
    }, f)

print("K-Means model saved: skin_kmeans.pkl")
print("Cluster centers (RGB):", kmeans_skin.cluster_centers_.astype(int))

In [ ]:
# ============================================================
# CELL 8: Evaluation
# ============================================================
checkpoint = torch.load('skin_tone_classifier.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(DEVICE))
        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=TONE_LABELS))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=TONE_LABELS,
            yticklabels=TONE_LABELS, cmap='OrRd')
plt.title('Confusion Matrix - Skin Tone Classifier (3 Class)')
plt.tight_layout()
plt.savefig('confusion_matrix_skin.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 9: Inference function
# ============================================================
def analyze_skin_tone(image_path, cnn_model, kmeans_data, device):
    """
    Returns skin tone class (Light/Medium/Dark), season, and color suggestions.
    """
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    cnn_model.eval()
    with torch.no_grad():
        out = cnn_model(tensor)
        probs = torch.softmax(out, dim=1)[0]
        pred_idx = probs.argmax().item()

    tone_label = kmeans_data['tone_labels'][pred_idx]  # 'Light'/'Medium'/'Dark'
    season = kmeans_data['season_map'][tone_label]
    colors = kmeans_data['color_suggestions'][season]

    return {
        'tone_class': tone_label,
        'confidence': float(probs[pred_idx]),
        'all_probs': {kmeans_data['tone_labels'][i]: float(probs[i]) for i in range(3)},
        'season': season,
        'recommended_colors': colors
    }

# Load kmeans
with open('skin_kmeans.pkl', 'rb') as f:
    kmeans_data = pickle.load(f)

# Test
test_img = val_df.iloc[0]['path']
result = analyze_skin_tone(test_img, model, kmeans_data, DEVICE)
print("Result:", json.dumps(result, indent=2))